# Tracksuit Take-Home Analysis

This notebook summarizes the baseline survey analysis and the proposed search/social triangulation extension.

## Framing

Goal: assess whether search-like signals can complement or partially substitute survey-based brand metrics, and identify where a text layer could improve interpretation.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
processed = ROOT / 'data' / 'processed'
panel = pd.read_csv(processed / 'brand_month_panel.csv', parse_dates=['wave_date'])
correlations = pd.read_csv(processed / 'metric_correlations.csv')
shortlist = pd.read_csv(processed / 'triangulation_shortlist.csv')
panel.head()

## Baseline read

The first pass uses the provided survey data only. It turns the raw file into a brand-month panel with:

- prompted awareness
- consideration
- preference
- funnel conversion rates
- within-category share metrics


In [ ]:
shortlist[['category_name', 'brand_count', 'month_count', 'triangulation_ready']].head(10)

## Categories that look most promising for triangulation

Current shortlist: Liquor Retailer, Garden Care, Household Cleaners (Indoor), Sugar confectionery, Audio Content.

These categories have enough history, enough brands, and enough spread in consideration to make external signals informative.

## Funnel alignment overview

Strongest current share-level alignment categories include: Online Outlet Retailer, Coffee, Chips, Crisps, Crackers, Haircare, Healthy Beverages.

![Funnel Correlation Overview](../figures/funnel_correlation_overview.png)

![Triangulation Shortlist](../figures/triangulation_shortlist.png)

## Example category reads

The two example time-series plots below show why this should be presented as category-conditional rather than universal.

![Liquor Retailer Consideration](../figures/liquor_retailer_consideration_timeseries.png)

![Household Cleaners Consideration](../figures/household_cleaners_consideration_timeseries.png)

## Google Trends merge

I pulled live Google Trends data for four shortlisted categories and merged it to the survey panel at the brand-month level.

The most important finding is not that search works everywhere. It is that search works unevenly:

- stronger in `Liquor Retailer`
- meaningfully positive in `Audio Content`
- mixed in `Social Media Platforms`
- weak in `Household Cleaners (Indoor)`

That pattern is exactly why a triangulation story is stronger than a one-metric replacement story.

In [ ]:
trends_merged = processed / 'survey_trends_merged.csv'
if trends_merged.exists():
    merged = pd.read_csv(trends_merged)
    rows = []
    for category_name, group in merged.groupby('category_name'):
        corr = group[['share_of_search','share_of_consideration','share_of_prompted_awareness','share_of_preference']].corr()
        rows.append({
            'category_name': category_name,
            'search_vs_share_consideration': corr.loc['share_of_search', 'share_of_consideration'],
            'search_vs_share_awareness': corr.loc['share_of_search', 'share_of_prompted_awareness'],
            'search_vs_share_preference': corr.loc['share_of_search', 'share_of_preference'],
        })
    display(pd.DataFrame(rows).sort_values('search_vs_share_consideration', ascending=False))
else:
    print('Run python3 scripts/fetch_google_trends.py after setting SERPAPI_API_KEY')

## Text and lexicon extension

The next layer is not intended to replace survey metrics directly. Instead, it helps interpret whether movement in search or mentions looks like:

- healthy demand and active consideration
- brand salience without clear purchase intent
- controversy or complaint-driven attention


In [ ]:
pd.read_csv(processed / 'sample_social_posts_scored.csv').head()

## Reddit pilot setup

For a lightweight Reddit pilot, I would prioritize three categories where text discussion is likely to be rich and decision-relevant:

- Household Cleaners (Indoor)
- Audio Content
- Social Media Platforms

The lexicon is category-aware, so terms like `mould`, `subscription`, `algorithm`, and `privacy` are only emphasized where they make sense.

In [ ]:
from pathlib import Path
reddit_scored = processed / 'reddit_pilot_template_scored.csv'
reddit_agg = processed / 'reddit_pilot_template_aggregated.csv'
if reddit_scored.exists() and reddit_agg.exists():
    display(pd.read_csv(reddit_scored).head())
    display(pd.read_csv(reddit_agg).head())
else:
    print('Run python3 scripts/score_external_text.py after filling data/external/reddit_pilot_template.csv')

## How I would use the Reddit pilot

I would not treat Reddit volume as a direct replacement for consideration. Instead, I would use it to explain whether a spike in search or salience is being driven by:

- shopping or switching intent
- product usage conversation
- complaints or controversy
- creator or platform buzz


## Recommendation direction

My current recommendation would be to position search as a fast directional signal for consideration in categories where it empirically aligns, then pair it with a lightweight discourse layer before making stronger brand-health claims. Survey data should remain the calibration anchor.